# Проверка сервиса BM25

Сервис **BM25** (код — ветка `dev/bm25`, развёрнут на сервере) ищет треки по словам: жанры, теги Last.fm,
артист + название. Отвечает списком треков (`ids`) с оценками (`scores`). Слова **английские**, как в каталоге.

Сам он ничего не рекомендует: это кандидаты, из которых пайплайн собирает ответ (`inference.ipynb`).
Описание API: `docs/candgen_api.md`.

Адрес и ключ — `BM25_URL`, `BM25_API_KEY` из файла `.env` в корне репозитория или секретов DataSphere.
Те же запросы можно делать в браузере: `<BM25_URL>/docs` (ключ — в поле `x-api-key`).

## Настройки

`DATA_DIR` — где лежит `tracks_meta`, из которого строились индексы (отсюда берутся названия треков).

In [ ]:
BM25_URL = "${BM25_URL}"    # из .env / окружения; можно вписать адрес прямо сюда
DATA_DIR = "CRS dataset"    # полный датасет: id с сервера — из него

In [ ]:
import os
import pandas as pd

if os.path.basename(os.getcwd()) == "examples":
    os.chdir("..")
from recsys.data.crs import parquet_files
from recsys.config import load_env
from recsys.retrieval.remote import CandgenClient

load_env()  # адреса и ключи из .env
bm25 = CandgenClient(BM25_URL, api_key="${BM25_API_KEY}")
print("BM25:", bm25.url)
columns = ["m4a_id", "m4a_artist", "m4a_song", "m4a_genres_full"]
tracks = pd.concat(pd.read_parquet(f, columns=columns) for f in parquet_files(DATA_DIR, "tracks_meta")).set_index("m4a_id")


def show(answer):
    """Ответ сервиса -> таблица: артист, песня, жанры, оценка."""
    return tracks.reindex(answer["ids"]).assign(score=answer["scores"])

## 1. Сервис работает?

Какие индексы загружены и сколько в них треков. Ошибка `Connection refused` — сервис не запущен.

In [ ]:
pd.DataFrame(bm25.get("/health")["indexes"]).T

## 2. BM25: поиск по словам

* `words` — английские слова, регистр не важен
* `k` — сколько треков вернуть (до 1000)
* `index` — где искать: `genres` (по умолчанию), `tags`, `title` (артист + название; на сервере пока нет — ответ 404)

Чем больше слов запроса у трека, тем выше оценка.

In [ ]:
show(bm25.post("/bm25/search", {"words": ["rock"], "k": 5, "index": "tags"}))

In [ ]:
show(bm25.post("/bm25/search", {"words": ["hard", "rock"], "k": 5, "index": "genres"}))

`exclude_ids` — какие треки не возвращать (пайплайн передаёт сюда уже показанные).

In [ ]:
first = bm25.post("/bm25/search", {"words": ["rock"], "k": 5, "index": "tags"})["ids"][0]
show(bm25.post("/bm25/search", {"words": ["rock"], "k": 5, "index": "tags", "exclude_ids": [first]}))

Русские слова не находятся: в каталоге английские теги. Перевод «рок» → `rock` делает пайплайн
при разборе запроса (`inference.ipynb`, раздел 5).

In [ ]:
bm25.post("/bm25/search", {"words": ["рок"], "k": 5, "index": "tags"})

## 3. Ошибки

| ответ | причина |
|---|---|
| `HTTP 404 Unknown index` | нет такого индекса |
| `HTTP 401` | нет ключа или он неверный (`BM25_API_KEY`) |
| `HTTP 422` | ошибка в теле запроса: лишнее поле, `k` > 1000 |
| `Connection refused`, таймаут | сервис не запущен или недоступен по сети |

In [ ]:
from recsys.retrieval.remote import CandgenError

try:
    bm25.post("/bm25/search", {"words": ["rock"], "k": 5, "index": "lyrics"})
except CandgenError as e:
    print(e)